In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# Resolve data paths relative to the repository, not the shell's current folder.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent


def data_path(*parts: str) -> Path:
    return PROJECT_ROOT.joinpath(*parts)

# 1. Check the Enriched / Aggregated Datasets
print("=== CHECKING TABLEAU_DATA / AGGREGATIONS ===")
try:
    df_sales = pd.read_csv(data_path("tableau_data", "fact_sales_daily.csv"))
    print("fact_sales_daily columns:", df_sales.columns.tolist())
    print("\nfact_sales_daily summary:")
    print(df_sales[['revenue', 'gross_profit']].describe())
    
    # Check negative gross profit instances
    neg_gp = df_sales[df_sales['gross_profit'] < 0]
    print(f"\nRows with negative Gross Profit: {len(neg_gp)} / {len(df_sales)} ({len(neg_gp)/len(df_sales)*100:.1f}%)")
except Exception as e:
    print(f"Error reading fact_sales_daily: {e}")

# 2. Check Raw Product Costs vs Prices
print("\n=== CHECKING RAW PRODUCTS & ORDER ITEMS ===")
try:
    df_prod = pd.read_csv(data_path("data", "products.csv"))
    print("products.csv columns:", df_prod.columns.tolist())
    
    # Look for cost vs price fields
    price_cols = [c for c in df_prod.columns if any(k in c.lower() for k in ['price', 'cost', 'retail', 'wholesale', 'margin'])]
    print("Price/Cost columns in products.csv:", price_cols)
    if price_cols:
        print(df_prod[price_cols].head())
except Exception as e:
    print(f"Error reading products.csv: {e}")

try:
    df_items = pd.read_csv(data_path("data", "order_items.csv"))
    print("\norder_items.csv columns:", df_items.columns.tolist())
    print(df_items.head(3))
except Exception as e:
    print(f"Error reading order_items.csv: {e}")

# 3. Check Promotions and Returns Impact
print("\n=== CHECKING PROMOS & RETURNS ===")
try:
    df_promo = pd.read_csv(data_path("data", "promotions.csv"))
    print("promotions.csv columns:", df_promo.columns.tolist())
    print(df_promo.head(3))
except Exception as e:
    print(f"Error reading promotions: {e}")

try:
    df_returns = pd.read_csv(data_path("data", "returns.csv"))
    print(f"Total return records: {len(df_returns)}")
except Exception as e:
    print(f"Error reading returns: {e}")

#4. Check Products & COGS
df_prod = pd.read_csv(data_path("data", "products.csv"))
df_prod["base_margin_pct"] = ((df_prod["price"] - df_prod["cogs"]) / df_prod["price"]) * 100
print(df_prod[["product_name", "price", "cogs", "base_margin_pct"]].describe())

=== CHECKING TABLEAU_DATA / AGGREGATIONS ===
fact_sales_daily columns: ['date', 'revenue', 'cogs', 'gross_profit', 'gross_margin_pct', 'day_of_week', 'is_weekend', 'month', 'day_of_year', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 'traffic_source', 'revenue_ma7', 'revenue_ma30']

fact_sales_daily summary:
            revenue  gross_profit
count  3.833000e+03  3.833000e+03
mean   4.286584e+06  5.914495e+05
std    2.624840e+06  6.661960e+05
min    2.798139e+05 -2.567312e+06
25%    2.471089e+06  2.292740e+05
50%    3.647304e+06  5.445544e+05
75%    5.350877e+06  8.760810e+05
max    2.090527e+07  4.369414e+06

Rows with negative Gross Profit: 382 / 3833 (10.0%)

=== CHECKING RAW PRODUCTS & ORDER ITEMS ===
products.csv columns: ['product_id', 'product_name', 'category', 'segment', 'size', 'color', 'price', 'cogs']
Price/Cost columns in products.csv: ['price']
          price
0  11059.650000
1   9523.076013
2  15951.633158
3  15753.717299
4  15766

/var/folders/6d/n0c7d2jd30sb50l09f4d_qsm0000gn/T/ipykernel_9742/2554608112.py:44: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(data_path("data", "order_items.csv"))


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "scripts":
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# ==========================================
# 1. FACT_SALES_DAILY.CSV
# ==========================================
print("\n" + "="*50 + "\n1. INSPECTING: fact_sales_daily.csv\n" + "="*50)
df_sales = pd.read_csv(PROJECT_ROOT / "tableau_data" / "fact_sales_daily.csv")
print("Shape:", df_sales.shape)
print("\nKey Metrics Summary:")
print(df_sales[['revenue', 'cogs', 'gross_profit', 'gross_margin_pct']].describe().T)

neg_days = df_sales[df_sales['gross_profit'] < 0]
print(f"\nTotal loss-making days: {len(neg_days)} ({len(neg_days)/len(df_sales)*100:.1f}%)")
print("Worst 3 loss days:\n", neg_days[['date', 'revenue', 'cogs', 'gross_profit', 'gross_margin_pct']].sort_values('gross_profit').head(3))

# ==========================================
# 2. DIM_PRODUCTS.CSV
# ==========================================
print("\n" + "="*50 + "\n2. INSPECTING: dim_products.csv\n" + "="*50)
df_prod = pd.read_csv(PROJECT_ROOT / "tableau_data" / "dim_products.csv")
print("Shape:", df_prod.shape)
print("Columns:", list(df_prod.columns))

# Margin & Return rates by Category
cat_summary = df_prod.groupby('category').agg(
    total_skus=('product_name', 'count'),
    avg_price=('price', 'mean'),
    avg_cogs=('cogs', 'mean'),
    avg_margin_pct=('profit_per_unit', lambda x: (x / df_prod.loc[x.index, 'price']).mean() * 100 if 'profit_per_unit' in df_prod.columns else 0),
    avg_return_rate=('return_rate_pct', 'mean') if 'return_rate_pct' in df_prod.columns else ('price', 'count')
).reset_index()
print("\nCategory Performance Breakdown:\n", cat_summary)

if 'profit_per_unit' in df_prod.columns:
    print("\nTop 3 Unprofitable / Lowest-Margin Products:\n", 
          df_prod.sort_values('profit_per_unit')[['product_name', 'category', 'price', 'cogs', 'profit_per_unit']].head(3))

# ==========================================
# 3. FACT_ORDERS_ENRICHED.CSV
# ==========================================
print("\n" + "="*50 + "\n3. INSPECTING: fact_orders_enriched.csv\n" + "="*50)
# Read sample or full
df_orders = pd.read_csv(PROJECT_ROOT / "tableau_data" / "fact_orders_enriched.csv", nrows=100000)
print("Previewing 100k rows sample. Columns:", list(df_orders.columns))

# Promo vs Non-Promo Performance
if 'has_promo' in df_orders.columns:
    promo_split = df_orders.groupby('has_promo').agg(
        order_count=('order_id', 'nunique'),
        avg_line_revenue=('line_revenue', 'mean'),
        avg_margin=('line_margin_pct', 'mean') if 'line_margin_pct' in df_orders.columns else ('line_revenue', 'count')
    ).reset_index()
    print("\nPromo Impact (0 = No Promo, 1 = With Promo):\n", promo_split)

# Channel Performance
if 'order_source' in df_orders.columns:
    chan_split = df_orders.groupby('order_source').agg(
        order_count=('order_id', 'nunique'),
        total_rev=('line_revenue', 'sum')
    ).reset_index()
    print("\nChannel Distribution:\n", chan_split)

# ==========================================
# 4. DIM_CUSTOMERS_RFM.CSV
# ==========================================
print("\n" + "="*50 + "\n4. INSPECTING: dim_customers_rfm.csv\n" + "="*50)
df_rfm = pd.read_csv(PROJECT_ROOT / "tableau_data" / "dim_customers_rfm.csv")
print("Shape:", df_rfm.shape)
print("\nCustomer Count & Avg Spend per RFM Segment:")
print(df_rfm.groupby('rfm_segment').agg(
    customer_count=('monetary', 'count'),
    avg_spend=('monetary', 'mean'),
    avg_recency=('recency_days', 'mean')
).sort_values('customer_count', ascending=False))

# ==========================================
# 5. AGG_COHORT_RETENTION.CSV
# ==========================================
print("\n" + "="*50 + "\n5. INSPECTING: agg_cohort_retention.csv\n" + "="*50)
df_cohort = pd.read_csv(PROJECT_ROOT / "tableau_data" / "agg_cohort_retention.csv")
print("Shape:", df_cohort.shape)
print("\nFirst 3 Cohorts (Month 0 to Month 6 Retention):\n", 
      df_cohort[df_cohort['period_number'].isin([0, 1, 2, 3, 6])].head(15))


1. INSPECTING: fact_sales_daily.csv
Shape: (3833, 17)

Key Metrics Summary:
                   count          mean           std         min         25%         50%         75%          max
revenue           3833.0  4.286584e+06  2.624840e+06   279813.94  2471088.82  3647303.90  5350877.20  20905271.35
cogs              3833.0  3.695134e+06  2.219789e+06   236576.31  2150580.23  3161112.99  4637293.92  16535857.67
gross_profit      3833.0  5.914495e+05  6.661960e+05 -2567311.72   229274.05   544554.38   876080.98   4369413.68
gross_margin_pct  3833.0  1.253931e+01  1.273556e+01      -57.46        8.26       17.83       20.29        28.69

Total loss-making days: 382 (10.0%)
Worst 3 loss days:
             date     revenue         cogs  gross_profit  gross_margin_pct
1151  2015-08-29  7794439.44  10361751.16   -2567311.72            -32.94
1150  2015-08-28  8506950.72  11044258.41   -2537307.69            -29.83
1883  2017-08-30  6946636.36   9396255.80   -2449619.44            -35.26


/var/folders/6d/n0c7d2jd30sb50l09f4d_qsm0000gn/T/ipykernel_9742/2633461519.py:52: DtypeWarning: Columns (0: promo_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_orders = pd.read_csv(PROJECT_ROOT / "tableau_data" / "fact_orders_enriched.csv", nrows=100000)


In [5]:
import pandas as pd

sales_df = pd.read_csv(
    data_path("data", "sales.csv"),
    parse_dates=["Date"],
)

print("Shape:", sales_df.shape)
print(f"Data starts on: {sales_df['Date'].min()}")
print(f"Data ends on:   {sales_df['Date'].max()}")

print("\nRow count per year:")
print(sales_df["Date"].dt.year.value_counts().sort_index())

print(f"\nMissing/Unparseable dates: {sales_df['Date'].isna().sum()}")

Shape: (3833, 3)
Data starts on: 2012-07-04 00:00:00
Data ends on:   2022-12-31 00:00:00

Row count per year:
Date
2012    181
2013    365
2014    365
2015    365
2016    366
2017    365
2018    365
2019    365
2020    366
2021    365
2022    365
Name: count, dtype: int64

Missing/Unparseable dates: 0
